[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/mobillity-univ/module-6-telling-the-story/notebook-6.5-from-colab-to-stakeholder-deck.ipynb)


# From a verified Colab analysis to a deck the transport authority can read

**The question.** We have already measured how often FGC trains run in the evening — one number per line, for the 20:00–24:00 window on a representative weekday. FGC is Ferrocarrils de la Generalitat de Catalunya, the operator behind Barcelona's metro, commuter-rail and funicular lines (there is no bus in this network). Next week that finding has to be presented to the transport authority. How do we turn an analysis that lives in code and charts into a short, professional slide deck — without rebuilding it slide by slide by hand?

**Why it's worth asking.** An analysis is only useful once the people who fund and plan the service can see it. The slow way is to open PowerPoint and retype every number by hand; the fast, reliable way is to keep the verified analysis as the single source and have an assistant write the code that assembles the deck for us.

**The data.** Each row of the timetable is a scheduled departure — a trip leaving a stop at a given time. From those rows we build one row per line: its evening departures, its departures per hour, and the longest a rider waits between trains in the evening. One honest caveat: these figures describe a single representative weekday and the evening window only, and they were checked against the real feed earlier — here we reuse that verified result, we don't re-audit it.

**The method — five steps, no slide touched by hand.** We hand the assistant the analysis, ask it for a presentation brief aimed at this audience, ask it for the code that builds the deck from that brief, run the code, and read the finished slides back.

In [1]:
# --- Setup: install the library this notebook uses -------------------------
# On Google Colab the first run installs python-pptx; run locally it is a no-op
# when the package is already present. Safe to re-run.
import importlib.util, subprocess, sys

if importlib.util.find_spec("pptx") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=True)
print("SETUP_OK: python-pptx available")

SETUP_OK: python-pptx available


## 1. Start from the analysis we already have

The analysis is done and checked. The wrong move now is to copy its numbers into slides one at a time — that is slow, and it is exactly where a figure gets mistyped. The right move is to hand the assistant the analysis itself, so every number on a slide traces straight back to the verified result and nothing is retyped. In a real session you attach the analysis notebook to the assistant as a file; here the verified finding is carried forward as a table so this notebook stands on its own. Before we hand it over, let's read what we are giving.

In [2]:
"""The verified per-line evening-service table - carried forward, not recomputed here.

One row per FGC line for a representative weekday evening (20:00-24:00), validated against
the real feed earlier in the course (story-5.4). Columns: line code, mode, evening
departures, departures/hour, worst-direction wait (min)."""
import pandas as pd

line_table = pd.DataFrame(
    [
        ("FV", "funicular", 80, 20.0, 6),
        ("L7", "metro", 56, 14.0, 9),
        ("L12", "metro", 45, 11.2, 11),
        ("L6", "metro", 39, 9.8, 13),
        ("S1", "rail", 25, 6.2, 24),
        ("S2", "rail", 25, 6.2, 24),
        ("L8", "metro", 16, 4.0, 34),
        ("R63", "rail", 3, 0.8, 80),
        ("S8", "rail", 14, 3.5, 80),
        ("R53", "rail", 2, 0.5, 120),
        ("S3", "rail", 5, 1.2, 120),
        ("S4", "rail", 4, 1.0, 120),
        ("R5", "rail", 7, 1.8, 240),
        ("R6", "rail", 6, 1.5, 240),
        ("RL1", "rail", 3, 0.8, 240),
    ],
    columns=["route_short_name", "mode", "evening_departures", "dep_per_hour", "evening_wait_min"],
)

print(line_table.to_string(index=False))
print("LINES:", line_table.route_short_name.nunique(),
      "| EVENING_TRIPS:", int(line_table.evening_departures.sum()))

route_short_name      mode  evening_departures  dep_per_hour  evening_wait_min
              FV funicular                  80          20.0                 6
              L7     metro                  56          14.0                 9
             L12     metro                  45          11.2                11
              L6     metro                  39           9.8                13
              S1      rail                  25           6.2                24
              S2      rail                  25           6.2                24
              L8     metro                  16           4.0                34
             R63      rail                   3           0.8                80
              S8      rail                  14           3.5                80
             R53      rail                   2           0.5               120
              S3      rail                   5           1.2               120
              S4      rail                   4      

Fifteen lines run in the evening, 330 departures in all. The spread is stark: the funicular, FV, comes every six minutes, while the R5, R6 and RL1 lines leave riders waiting up to 240 minutes — four hours — between trains. That unevenness is the story the deck has to tell, and it is now sitting in a table every slide can point at.

## 2. Ask the assistant for a presentation brief

A notebook full of numbers is not a presentation. Before any slides exist, we ask the assistant — with the analysis in hand — to turn the finding into a presentation brief: the plan for the deck, written out. We do not ask vaguely for "a presentation"; a vague ask leaves the assistant to guess who the deck is for and what it should argue, and it comes back as a tidy slide-dump of the table. Instead we spell out the parts that decide how the deck reads — the role it should take, the audience, the goal, the tone and format, and the structure — so the brief comes back aimed at the right room. We ask for the message as a pyramid: the recommendation first, then the evidence, then the action. A prompt for that:

Here is my verified analysis of FGC evening service — one row per line for a weekday evening (20:00–24:00): each line's evening departures, departures per hour, and the longest a rider waits between trains in the worst direction. Turn it into a presentation brief.

```markdown
**Role:** You are a transit analyst writing a presentation brief.
**Audience:** The regional transport authority — the people who fund and plan service, not analysts.
**Goal:** Lead them to a decision on where to add evening service, starting with the lines riders wait longest for.
**Context:** The analysis covers FGC evening service on a weekday, 20:00-24:00 — each line's departures and the worst-direction wait between trains. Base every figure on it; invent nothing.
**Format & Tone:** A slide-ready brief I can hand straight to a slide builder — about seven slides, one titled point per slide (a bold title and a single supporting line). Executive and plain: no jargon, no long paragraphs. Note the look I want: clean slides, one accent colour, body text large enough to read from the back of a room, and the evening-frequency chart on the evidence slide.
**Structure:** Minto's Pyramid — the recommendation-first conclusion, then the supporting evidence, then the action to take.

line  mode        evening_departures  dep_per_hour  wait_min
FV    funicular   80                  20.0          6
L7    metro       56                  14.0          9
L12   metro       45                  11.2          11
L6    metro       39                  9.8           13
S1    rail        25                  6.2           24
S2    rail        25                  6.2           24
L8    metro       16                  4.0           34
R63   rail        3                   0.8           80
S8    rail        14                  3.5           80
R53   rail        2                   0.5           120
S3    rail        5                   1.2           120
S4    rail        4                   1.0           120
R5    rail        7                   1.8           240
R6    rail        6                   1.5           240
RL1   rail        3                   0.8           240
```

The brief comes back like this:

```markdown
# FGC Evening Service — Presentation Brief
For the regional transport authority. Tone: executive, plain, decision-first. Format: a shareable PowerPoint, seven slides, one titled point per slide — a bold title and a single supporting line. Look: clean white slides, one blue accent, body text readable from the back of the room, and the frequency chart on the evidence slide. Every figure is from the analysis; no new numbers. Structure: Minto's Pyramid — conclusion, then evidence, then action.

**Slide 1 — FGC Evening Service Frequency:** Weekday evening, 20:00-24:00. Prepared for the transport authority.
**Slide 2 — The bottom line:** Evening service is uneven — riders on the thinner-served lines wait far longer for a train than riders on the core network.
**Slide 3 — Evidence:** Minutes between evening trains, by line.
**Slide 4 — The gap is large:** The best-served line runs every 6 minutes; the worst waits 240 minutes — 40 times longer.
**Slide 5 — Who is affected:** Six lines wait two hours or more between evening trains — R53, S3, S4, R5, R6 and RL1.
**Slide 6 — What it means:** Riders on the outer rail lines face far longer evening waits than the core metro network.
**Slide 7 — Recommendation:** Add evening trips on the lines waiting two hours or more, then re-check after the next timetable change.
```

That is a real brief, not a slide-dump: it opens with the recommendation, every slide is one titled point a decision-maker can check, and because we named the audience, the tone and the look, it is detailed enough that a person — or the assistant itself — could build the deck straight from it. Seven slides, every figure traceable to the analysis. This brief is the spec the next step turns into code.

## 3. Ask the assistant to write the code that builds the deck

Now the format. The module covered several ways to generate slides from code — a PowerPoint file with python-pptx, a web deck with reveal.js, quick Marp markdown, academic Beamer. The choice is driven by the audience: the authority wants a file they can open, forward, and present in a meeting, so we take the PowerPoint route. This is the moment it is tempting to open PowerPoint and lay the slides out by hand — and that is the move to skip. The brief and the numbers already exist, so instead we hand the assistant the brief and ask it to write the python-pptx code that turns it into the deck.

Using `python-pptx`, write Python that builds this brief into a PowerPoint deck.

```markdown
Build one slide per titled point: the title in bold with a blue accent, and the supporting line below it in 24-point text. Render the evening-frequency chart from the finding and put it on the evidence slide. Then save the file.
```

## 4. Run the code and build the deck

We run the code the assistant wrote. It renders the evening-frequency chart from the finding, then reads the brief we agreed on — written down here as the list of slides so the code can loop over it — lays out one slide per titled point in the look the brief asked for, drops the chart onto the evidence slide, and writes the file.

In [3]:
"""
Build the PowerPoint deck from the brief.
Data: the brief (one titled point per slide) + line_table (for the evidence chart).
"""
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 0. Settings you can change
DECK = "story-6.5-fgc-evening-service.pptx"
CHART = "story-6.5-evening-frequency.png"
ACCENT = RGBColor(0x25, 0x63, 0xEB)   # one blue accent, as the brief asked
BODY_PT = 24                          # readable from the back of the room

# 1. The brief, written down as (title, supporting line) per slide
brief = [
    ("FGC Evening Service Frequency", "Weekday evening, 20:00-24:00. Prepared for the transport authority."),
    ("The bottom line", "Evening service is uneven — riders on the thinner-served lines wait far longer for a train than riders on the core network."),
    ("Evidence", "Minutes between evening trains, by line."),
    ("The gap is large", "The best-served line runs every 6 minutes; the worst waits 240 minutes — 40 times longer."),
    ("Who is affected", "Six lines wait two hours or more between evening trains: R53, S3, S4, R5, R6 and RL1."),
    ("What it means", "Riders on the outer rail lines face far longer evening waits than the core metro network."),
    ("Recommendation", "Add evening trips on the lines waiting two hours or more, then re-check after the next timetable change."),
]

# 2. Render the evidence chart from the finding
ranked = line_table.sort_values("evening_wait_min")
colors = ["#10B981" if w <= 15 else "#F59E0B" if w < 120 else "#DC2626" for w in ranked.evening_wait_min]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranked.route_short_name, ranked.evening_wait_min, color=colors)
ax.invert_yaxis()
ax.set_xlabel("Minutes between trains - worst direction, evening 20:00-24:00")
ax.set_title("FGC evening service is uneven by line")
for i, w in enumerate(ranked.evening_wait_min):
    ax.text(w + 3, i, f"{int(w)}m", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(CHART, dpi=120)
plt.close(fig)

# 3. Build one slide per titled point, in the look the brief asked for
prs = Presentation()
blank = prs.slide_layouts[6]
for title_text, line in brief:
    slide = prs.slides.add_slide(blank)
    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(9), Inches(1)).text_frame
    title.text = title_text
    title_run = title.paragraphs[0].runs[0]
    title_run.font.size = Pt(32)
    title_run.font.bold = True
    title_run.font.color.rgb = ACCENT
    body = slide.shapes.add_textbox(Inches(0.5), Inches(1.7), Inches(9), Inches(4)).text_frame
    body.word_wrap = True
    body.text = line
    body.paragraphs[0].runs[0].font.size = Pt(BODY_PT)
    if title_text == "Evidence":
        slide.shapes.add_picture(CHART, Inches(0.5), Inches(2.6), width=Inches(8))

# 4. Save the deck
prs.save(DECK)
print("CHART_SAVED:", CHART)
print("DECK_SAVED:", DECK)
print("SLIDE_COUNT:", len(prs.slides._sldIdLst))

CHART_SAVED: story-6.5-evening-frequency.png
DECK_SAVED: story-6.5-fgc-evening-service.pptx
SLIDE_COUNT: 7


It saves cleanly: seven slides, the evidence chart embedded on its slide, every number carried straight from the verified table — nothing retyped. The deck was assembled entirely by code from the brief, which is exactly what lets us trust the figures on it.

## 5. Read the finished presentation

The last step is to look at what we built the way the audience will. Generated code can quietly drop a slide, repeat one, or skip the chart, and it is far cheaper to catch that here than in front of the authority. So we open the saved deck and read its slide titles back, in order, to confirm the deck matches the brief — the same quick check you would give any generated file before it leaves your screen.

In [4]:
"""
Read the finished deck back: the slides it built, in order.
Data: the saved .pptx.
"""
from pptx import Presentation

deck = Presentation(DECK)
titles = [slide.shapes[0].text_frame.text for slide in deck.slides]
for i, title in enumerate(titles, start=1):
    print(f"Slide {i}: {title}")
print("\nSLIDES_MATCH_BRIEF:", titles == [t for t, _ in brief])

Slide 1: FGC Evening Service Frequency
Slide 2: The bottom line
Slide 3: Evidence
Slide 4: The gap is large
Slide 5: Who is affected
Slide 6: What it means
Slide 7: Recommendation

SLIDES_MATCH_BRIEF: True


And there is the deck: seven slides, in the order the brief laid out, headed by the recommendation and carrying the evidence chart. We started with a finding that lived in code and charts — FGC evening service is uneven, with the outer rail lines waiting up to four hours between trains — due in front of the transport authority next week. By handing the assistant the analysis, asking it for a brief, asking it for the code, and running that code, we turned it into a deck the authority can open, with every figure carried straight from the verified analysis and no slide touched by hand. That is the whole path, and it is the one to reuse for any analysis that has to leave your screen and reach the people who decide what happens next.